In [5]:
import pandas as pd
import numpy as np
from collections import Counter

In [8]:
def preprocess_rich(
    symbol: str,
    lag: int = 30,
    use_calendar: bool = True,
    windows=(3,5,10,20,30),
    use_fft=True,
    verbose: bool = False,
):
    """
    Trả về dict {'train': (X_train, y_train), 'test': (X_test, y_test)}.
    - KHÔNG dùng OHLC ngày t cho dự đoán ngày t: chỉ dùng diff của chuỗi đã shift(1).
    - Đặc trưng gồm: lag của diff (0..lag), rolling stats (mean/std/min/max/skew/kurt),
      tỉ lệ ngày tăng (upratio) trên close/open, label_lag_1..5 + streak, tương tác đơn giản,
      FFT band-power trên close_diff (tùy chọn), lịch (sin/cos) (tùy chọn).
    - Để tránh rụng toàn bộ do NaN từ rolling/FFT, fillna(0) (tree-friendly).
    """
    train_path = f"F:\\EL4TF\\data\\vn30\\binary\\{symbol}_train.csv"
    test_path  = f"F:\\EL4TF\\data\\vn30\\binary\\{symbol}_test.csv"
    df_train = pd.read_csv(train_path)
    df_test  = pd.read_csv(test_path)

    # map label
    label_mapping = {"up": 1, "down": 0}
    for df in (df_train, df_test):
        df["label"] = df["label"].map(label_mapping)
        if "volume" in df.columns:
            df.drop(columns=["volume"], inplace=True, errors="ignore")

    test_size = len(df_test)
    df_all = pd.concat([df_train, df_test], ignore_index=True)

    # shift(1) rồi mới diff => diff tại chỉ số t là biến động của (t-1) so (t-2)
    for col in ["open", "high", "low", "close"]:
        df_all[f"{col}_shift"] = df_all[col].shift(1)
        df_all[f"{col}_diff"]  = df_all[f"{col}_shift"].diff()

    diff_cols = [f"{c}_diff" for c in ["open", "high", "low", "close"]]

    # lag block cho diff
    lag_frames = {}
    for c in diff_cols:
        s = df_all[c]
        for i in range(0, lag+1):
            lag_frames[f"{c}_lag_{i}"] = s.shift(i)
    lag_block = pd.DataFrame(lag_frames, index=df_all.index)

    # rolling stats trên diff
    roll_feats = {}
    for c in diff_cols:
        s = df_all[c]
        for w in windows:
            r = s.rolling(window=w, min_periods=w)
            roll_feats[f"{c}_mean_{w}"] = r.mean()
            roll_feats[f"{c}_std_{w}"]  = r.std()
            roll_feats[f"{c}_min_{w}"]  = r.min()
            roll_feats[f"{c}_max_{w}"]  = r.max()
            roll_feats[f"{c}_skew_{w}"] = r.skew()
            roll_feats[f"{c}_kurt_{w}"] = r.kurt()
            mu = r.mean(); sd = r.std()
            roll_feats[f"{c}_zlast_{w}"] = (s - mu) / (sd.replace(0, np.nan))
    roll_block = pd.DataFrame(roll_feats, index=df_all.index)

    # tỉ lệ ngày tăng (upratio)
    updown_feats = {}
    for base_c in ["close_diff", "open_diff"]:
        pos = (df_all[base_c] > 0).astype(float)
        for w in windows:
            updown_feats[f"{base_c}_upratio_{w}"] = pos.rolling(w, min_periods=w).mean()
    updown_block = pd.DataFrame(updown_feats, index=df_all.index)

    # label lag + streak
    lbl_shift1 = df_all["label"].shift(1)
    sgn = np.where(lbl_shift1==1, 1, np.where(lbl_shift1==0, -1, 0))
    streak = np.zeros_like(sgn, dtype=float)
    cur = 0
    for i, v in enumerate(sgn):
        if v==0:
            streak[i] = cur
        else:
            cur = cur+1 if (cur>0 and v>0) else (cur-1 if (cur<0 and v<0) else (1 if v>0 else -1))
            streak[i] = cur
    streak_block = pd.DataFrame({
        "label_lag_1": lbl_shift1,
        "label_lag_2": df_all["label"].shift(2),
        "label_lag_3": df_all["label"].shift(3),
        "label_lag_4": df_all["label"].shift(4),
        "label_lag_5": df_all["label"].shift(5),
        "streak_signed": streak,
    }, index=df_all.index)

    # tương tác đơn giản trên diff
    inter_block = pd.DataFrame({
        "range_diff": (df_all["high_diff"] - df_all["low_diff"]),
        "body_diff":  (df_all["close_diff"] - df_all["open_diff"]),
    }, index=df_all.index)
    inter_block["abs_range_diff"] = inter_block["range_diff"].abs()
    inter_block["abs_body_diff"]  = inter_block["body_diff"].abs()

    # FFT band-power (nhẹ nhàng, chỉ close_diff và w=20,30)
    def _fft_bandpowers(x, bands=(0.25,0.5,0.75)):
        n = len(x)
        if n < 4: return (np.nan, np.nan, np.nan)
        spec = np.fft.rfft(x - x.mean())
        p = (spec.real**2 + spec.imag**2)[1:]
        if p.size == 0 or p.sum() == 0: return (np.nan, np.nan, np.nan)
        m = p.size
        b1, b2, b3 = int(m*bands[0]), int(m*bands[1]), int(m*bands[2])
        low, mid, high = p[:b1].sum(), p[b1:b2].sum(), (p[b2:b3].sum() + p[b3:].sum())
        tot = low+mid+high
        if tot == 0: return (np.nan, np.nan, np.nan)
        return (low/tot, mid/tot, high/tot)

    fft_block = {}
    if use_fft:
        arr = df_all["close_diff"].to_numpy()
        for w in [20, 30]:
            lows, mids, highs = [], [], []
            for i in range(len(df_all)):
                if i+1 >= w:
                    pl, pm, ph = _fft_bandpowers(arr[i-w+1:i+1])
                else:
                    pl, pm, ph = (np.nan, np.nan, np.nan)
                lows.append(pl); mids.append(pm); highs.append(ph)
            fft_block[f"close_fft_low_{w}"]  = lows
            fft_block[f"close_fft_mid_{w}"]  = mids
            fft_block[f"close_fft_high_{w}"] = highs
    fft_block = pd.DataFrame(fft_block, index=df_all.index) if len(fft_block)>0 else None

    # calendar (cyclic)
    cal_block = None
    if use_calendar and "time" in df_all.columns:
        t = pd.to_datetime(df_all["time"])
        dow = t.dt.dayofweek; dom = t.dt.day; doy = t.dt.dayofyear
        cal_block = pd.DataFrame({
            "sin_dow": np.sin(2*np.pi*dow/7),   "cos_dow": np.cos(2*np.pi*dow/7),
            "sin_dom": np.sin(2*np.pi*dom/31),  "cos_dom": np.cos(2*np.pi*dom/31),
            "sin_doy": np.sin(2*np.pi*doy/366), "cos_doy": np.cos(2*np.pi*doy/366),
        }, index=df_all.index)

    # ghép block một lần để tránh fragmentation
    blocks = [lag_block, roll_block, updown_block, streak_block, inter_block]
    if fft_block is not None: blocks.append(fft_block)
    if cal_block is not None: blocks.append(cal_block)
    X_all = pd.concat(blocks, axis=1).fillna(0)  # fillna(0) để không rụng hàng

    y_all = df_all["label"]

    # chia train/test theo thời gian
    X_train, X_test = X_all.iloc[:-test_size], X_all.iloc[-test_size:]
    y_train, y_test = y_all.iloc[:-test_size], y_all.iloc[-test_size:]

    if verbose:
        print(f"=== Preprocess {symbol} (rich) ===")
        print(f"Train: {X_train.shape} | Test: {X_test.shape}")
        print(f"Label dist train: {Counter(y_train)}, test: {Counter(y_test)}")

    return {"train": (X_train, y_train), "test": (X_test, y_test)}

In [9]:
data_base = preprocess_rich(symbol="ACB")

In [11]:
# Try class_weight='balanced' on base features with simple RF hyperparams
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score

X_train_full, y_train_full = data_base["train"]
X_test, y_test = data_base["test"]

rf_cw = RandomForestClassifier(
    n_estimators=120,
    max_depth=12,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features="sqrt",
    bootstrap=True,
    class_weight="balanced",
    random_state=11,
    n_jobs=1
)
rf_cw.fit(X_train_full, y_train_full)
y_pred = rf_cw.predict(X_test)
float(balanced_accuracy_score(y_test, y_pred))


0.4948272642390289

In [12]:
import numpy as np
import pandas as pd
from typing import Tuple, Dict
from collections import Counter
from scipy.stats import ks_2samp

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report

# =============== Base features ===============
def build_features(symbol: str = "ACB", lag: int = 30, add_ewm: bool = True) -> Dict[str, Tuple[pd.DataFrame, pd.Series]]:
    df_train = pd.read_csv(f"./{symbol}_train.csv")
    df_test  = pd.read_csv(f"./{symbol}_test.csv")

    # label map
    label_mapping = {"up": 1, "down": 0}
    for df in (df_train, df_test):
        df["label"] = df["label"].map(label_mapping)
        if "volume" in df.columns:
            df.drop(columns=["volume"], inplace=True, errors="ignore")

    test_size = len(df_test)
    df_all = pd.concat([df_train, df_test], ignore_index=True)

    # shift(1) rồi mới diff => không dùng OHLC của ngày dự đoán
    for col in ["open", "high", "low", "close"]:
        df_all[f"{col}_shift"] = df_all[col].shift(1)
        df_all[f"{col}_diff"]  = df_all[f"{col}_shift"].diff()

    # lag cho diff
    diff_cols = [f"{c}_diff" for c in ["open", "high", "low", "close"]]
    feats = {}
    for c in diff_cols:
        s = df_all[c]
        for i in range(0, lag+1):
            feats[f"{c}_lag_{i}"] = s.shift(i)
    X_all = pd.DataFrame(feats, index=df_all.index)

    # EWMA rất nhẹ trên close_diff (ổn định hơn rolling thông thường)
    if add_ewm:
        s = df_all["close_diff"]
        for span in [5, 10, 20]:
            X_all[f"close_ewm_mean_{span}"] = s.ewm(span=span, min_periods=span, adjust=False).mean()
            X_all[f"close_ewm_std_{span}"]  = s.ewm(span=span, min_periods=span, adjust=False).std()

    y_all = df_all["label"]

    # split theo thời gian
    X_train, X_test = X_all.iloc[:-test_size], X_all.iloc[-test_size:]
    y_train, y_test = y_all.iloc[:-test_size], y_all.iloc[-test_size:]

    # bỏ hàng bị NaN do lag/ewm
    mask_tr = X_train.notna().all(axis=1); mask_te = X_test.notna().all(axis=1)
    X_train, y_train = X_train[mask_tr], y_train[mask_tr]
    X_test,  y_test  = X_test[mask_te],  y_test[mask_te]

    return {"train": (X_train, y_train), "test": (X_test, y_test)}

# =============== Distribution shift tools ===============
def diag_align_train_to_test(X_train: pd.DataFrame, X_test: pd.DataFrame) -> pd.DataFrame:
    """(x - mu_tr)/std_tr * std_te + mu_te (áp dụng cho train; test giữ nguyên)."""
    mu_tr = X_train.mean(axis=0); std_tr = X_train.std(axis=0).replace(0, 1.0)
    mu_te = X_test.mean(axis=0);  std_te = X_test.std(axis=0).replace(0, 1.0)
    Xc = (X_train - mu_tr) / std_tr
    Xa = Xc * std_te + mu_te
    return Xa

def domain_importance_weights(X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: pd.Series,
                              time_decay_alpha: float = 1.0, clip=(0.02, 0.98)) -> np.ndarray:
    """w = odds(test|x) * time_decay * class_balance, cắt outlier và chuẩn hoá."""
    scaler = StandardScaler(with_mean=True, with_std=True)
    X_comb = np.vstack([X_train.values, X_test.values]); scaler.fit(X_comb)
    X_tr = scaler.transform(X_train.values); X_te = scaler.transform(X_test.values)

    y_dom = np.hstack([np.zeros(len(X_train)), np.ones(len(X_test))])
    dom = LogisticRegression(max_iter=500, class_weight="balanced", n_jobs=1).fit(
        np.vstack([X_tr, X_te]), y_dom
    )
    p_test = np.clip(dom.predict_proba(X_tr)[:,1], 1e-3, 1-1e-3)
    iw = p_test/(1-p_test)

    n = len(X_train); t = np.arange(n)/max(1, n-1)
    td = np.exp(time_decay_alpha * t)

    n0 = (y_train==0).sum(); n1 = (y_train==1).sum(); N=len(y_train)
    cw0 = N/(2*n0); cw1 = N/(2*n1)
    class_bal = np.where(y_train.values==1, cw1, cw0)

    w = iw * td * class_bal
    lo, hi = np.quantile(w, clip[0]), np.quantile(w, clip[1])
    w = np.clip(w, lo, hi); w = w/np.median(w)
    return w

def ks_stable_cols(X_train: pd.DataFrame, X_test: pd.DataFrame, topk: int = 80):
    """Chọn top-k feature có KS(train,test) nhỏ nhất (ổn định nhất)."""
    scores = {c: ks_2samp(X_train[c].values, X_test[c].values).statistic for c in X_train.columns}
    cols = sorted(scores.keys(), key=lambda c: scores[c])[:topk]
    return cols

# =============== Train & evaluate ===============
def split_tail_validation(X_train: pd.DataFrame, y_train: pd.Series, val_frac: float = 0.2):
    n = len(X_train); n_tr = int(n*(1-val_frac))
    return (X_train.iloc[:n_tr], y_train.iloc[:n_tr]), (X_train.iloc[n_tr:], y_train.iloc[n_tr:])

def run_pipeline(symbol="ACB", lag=30, add_ewm=True, use_align=True, use_importance=True,
                 use_ks=True, ks_topk=80, tail_frac=None):
    data = build_features(symbol, lag=lag, add_ewm=add_ewm)
    X_train, y_train = data["train"]; X_test, y_test = data["test"]

    # dùng đoạn train gần hiện tại (giảm lỗi thời của quy luật cũ)
    if tail_frac is not None and 0 < tail_frac < 1.0:
        n = len(X_train); start = int(n*(1-tail_frac))
        X_train = X_train.iloc[start:]; y_train = y_train.iloc[start:]

    # chọn feature ổn định
    if use_ks:
        cols = ks_stable_cols(X_train, X_test, topk=ks_topk)
        X_train, X_test = X_train[cols], X_test[cols]

    # căn chỉnh phân phối train -> test
    X_train_aligned = diag_align_train_to_test(X_train, X_test) if use_align else X_train

    # sample weights (reweight + time-decay + class balance)
    if use_importance:
        w = domain_importance_weights(X_train_aligned, X_test, y_train, time_decay_alpha=1.0, clip=(0.02,0.98))
    else:
        n0 = (y_train==0).sum(); n1 = (y_train==1).sum(); N=len(y_train)
        cw0 = N/(2*n0); cw1 = N/(2*n1)
        w = np.where(y_train.values==1, cw1, cw0)

    # models
    out = {}

    rf = RandomForestClassifier(
        n_estimators=240, max_depth=16, min_samples_split=4, min_samples_leaf=2,
        max_features="sqrt", bootstrap=True, random_state=2025, n_jobs=-1
    ).fit(X_train_aligned, y_train, sample_weight=w)
    y_rf = rf.predict(X_test)
    out["rf_bacc"] = float(balanced_accuracy_score(y_test, y_rf))

    et = ExtraTreesClassifier(
        n_estimators=500, max_depth=None, min_samples_split=2, min_samples_leaf=1,
        max_features="sqrt", random_state=7, n_jobs=-1
    ).fit(X_train_aligned, y_train, sample_weight=w)
    y_et = et.predict(X_test)
    out["et_bacc"] = float(balanced_accuracy_score(y_test, y_et))

    # HistGB: OOD khá ổn, dùng sample_weight
    try:
        hgb = HistGradientBoostingClassifier(
            max_depth=6, learning_rate=0.06, max_iter=800, min_samples_leaf=20,
            l2_regularization=0.0, early_stopping=True, random_state=19
        ).fit(X_train_aligned, y_train, sample_weight=w)
        y_hgb = hgb.predict(X_test)
        out["hgb_bacc"] = float(balanced_accuracy_score(y_test, y_hgb))
    except Exception as e:
        out["hgb_bacc"] = None
        out["hgb_err"] = str(e)

    # ensemble xác suất (equal weights hoặc học trọng số trên validation tail)
    proba_rf = rf.predict_proba(X_test)[:,1]
    proba_et = et.predict_proba(X_test)[:,1]

    # học trọng số trộn trên validation tail
    (X_tr, y_tr), (X_val, y_val) = split_tail_validation(X_train_aligned, y_train, val_frac=0.2)
    prf_val = rf.predict_proba(X_val)[:,1]
    pet_val = et.predict_proba(X_val)[:,1]
    val_mat = np.vstack([prf_val, pet_val]).T
    yv = y_val.values.astype(float)

    lam = 1e-3
    A = val_mat.T @ val_mat + lam*np.eye(val_mat.shape[1]); b = val_mat.T @ yv
    w_raw = np.linalg.solve(A, b); w_raw = np.clip(w_raw, 0, None)
    w_blend = w_raw/w_raw.sum() if w_raw.sum()>0 else np.ones_like(w_raw)/len(w_raw)

    proba_blend = w_blend[0]*proba_rf + w_blend[1]*proba_et
    if out.get("hgb_bacc") is not None:
        phgb_val = hgb.predict_proba(X_val)[:,1]
        val_mat = np.vstack([prf_val, pet_val, phgb_val]).T
        A = val_mat.T @ val_mat + lam*np.eye(val_mat.shape[1]); b = val_mat.T @ yv
        w_raw = np.linalg.solve(A, b); w_raw = np.clip(w_raw, 0, None)
        w_blend = w_raw/w_raw.sum() if w_raw.sum()>0 else np.ones_like(w_raw)/len(w_raw)
        proba_blend = (w_blend[0]*proba_rf + w_blend[1]*proba_et + w_blend[2]*hgb.predict_proba(X_test)[:,1])

    y_blend = (proba_blend >= 0.5).astype(int)
    out["blend_weights"] = w_blend.tolist()
    out["blend_bacc"] = float(balanced_accuracy_score(y_test, y_blend))
    out["confusion_blend"] = confusion_matrix(y_test, y_blend).tolist()
    out["report_blend"] = classification_report(y_test, y_blend, digits=4)

    return out

if __name__ == "__main__":
    cfgs = [
        dict(lag=30, add_ewm=True,  use_align=True,  use_importance=True,  use_ks=True, ks_topk=80, tail_frac=None),
        dict(lag=30, add_ewm=True,  use_align=True,  use_importance=True,  use_ks=True, ks_topk=60, tail_frac=0.6),
        dict(lag=20, add_ewm=True,  use_align=True,  use_importance=True,  use_ks=True, ks_topk=60, tail_frac=0.6),
        dict(lag=40, add_ewm=False, use_align=True,  use_importance=True,  use_ks=True, ks_topk=100,tail_frac=None),
        dict(lag=30, add_ewm=False, use_align=False, use_importance=False, use_ks=False, ks_topk=0,  tail_frac=None),
    ]
    for i,cfg in enumerate(cfgs, 1):
        print(f"\n=== EXP #{i}: {cfg} ===")
        out = run_pipeline(**cfg)
        print(out)


=== EXP #1: {'lag': 30, 'add_ewm': True, 'use_align': True, 'use_importance': True, 'use_ks': True, 'ks_topk': 80, 'tail_frac': None} ===
{'rf_bacc': 0.525060690943044, 'et_bacc': 0.5148272642390289, 'hgb_bacc': 0.480578898225957, 'blend_weights': [0.0013906864082278499, 0.9986093135917721, 0.0], 'blend_bacc': 0.5152380952380953, 'confusion_blend': [[122, 53], [102, 51]], 'report_blend': '              precision    recall  f1-score   support\n\n           0     0.5446    0.6971    0.6115       175\n           1     0.4904    0.3333    0.3969       153\n\n    accuracy                         0.5274       328\n   macro avg     0.5175    0.5152    0.5042       328\nweighted avg     0.5193    0.5274    0.5114       328\n'}

=== EXP #2: {'lag': 30, 'add_ewm': True, 'use_align': True, 'use_importance': True, 'use_ks': True, 'ks_topk': 60, 'tail_frac': 0.6} ===
{'rf_bacc': 0.483828197945845, 'et_bacc': 0.49320261437908497, 'hgb_bacc': 0.4687394957983193, 'blend_weights': [0.00217548158248617

In [13]:
# === Block ①: Utils & Preprocess (shift->diff->lag, NO leakage) ===
import os
import numpy as np
import pandas as pd
from typing import Dict, Tuple, List
from collections import Counter
from scipy.stats import ks_2samp

DATA_DIR = "."   # đổi thành "." nếu chạy local cùng thư mục

def load_raw(symbol: str = "ACB") -> Tuple[pd.DataFrame, pd.DataFrame]:
    df_train = pd.read_csv(os.path.join(DATA_DIR, f"{symbol}_train.csv"))
    df_test  = pd.read_csv(os.path.join(DATA_DIR, f"{symbol}_test.csv"))
    # map label
    label_mapping = {"up": 1, "down": 0}
    for df in (df_train, df_test):
        df["label"] = df["label"].map(label_mapping)
        if "volume" in df.columns:
            df.drop(columns=["volume"], inplace=True, errors="ignore")
    return df_train, df_test

def build_base_features(df_train: pd.DataFrame, df_test: pd.DataFrame, lag: int = 30) -> Dict[str, Tuple[pd.DataFrame, pd.Series]]:
    """
    Tạo đặc trưng theo đúng yêu cầu:
    - Chỉ dùng chuỗi đã shift(1) rồi diff (bậc 1)
    - Sau đó tạo các lag 0..lag cho {open,high,low,close}_diff
    """
    test_size = len(df_test)
    df_all = pd.concat([df_train, df_test], ignore_index=True)

    for col in ["open", "high", "low", "close"]:
        df_all[f"{col}_shift"] = df_all[col].shift(1)
        df_all[f"{col}_diff"]  = df_all[f"{col}_shift"].diff()

    diff_cols = [f"{c}_diff" for c in ["open", "high", "low", "close"]]
    feats = {}
    for c in diff_cols:
        s = df_all[c]
        for i in range(0, lag+1):
            feats[f"{c}_lag_{i}"] = s.shift(i)

    X_all = pd.DataFrame(feats, index=df_all.index)
    y_all = df_all["label"]

    # split theo thời gian
    X_train, X_test = X_all.iloc[:-test_size], X_all.iloc[-test_size:]
    y_train, y_test = y_all.iloc[:-test_size], y_all.iloc[-test_size:]

    # loại NaN do lag ở đầu dãy
    mask_tr = X_train.notna().all(axis=1)
    mask_te = X_test.notna().all(axis=1)
    X_train, y_train = X_train[mask_tr], y_train[mask_tr]
    X_test,  y_test  = X_test[mask_te],  y_test[mask_te]

    print(f"[Preprocess] Train: {X_train.shape} | Test: {X_test.shape}")
    print(f"[Preprocess] Label dist train: {Counter(y_train)}, test: {Counter(y_test)}")

    return {"train": (X_train, y_train), "test": (X_test, y_test)}

# --- Khoảng cách phân phối (KS & PSI) ---
def ks_distance_matrix(Xa: pd.DataFrame, Xb: pd.DataFrame) -> float:
    """Trả về KS trung bình (càng nhỏ càng "gần")."""
    vals = []
    for c in Xa.columns:
        try:
            vals.append(ks_2samp(Xa[c].values, Xb[c].values).statistic)
        except Exception:
            continue
    return float(np.nanmean(vals)) if len(vals) else np.nan

def population_stability_index(X_train: pd.DataFrame, X_test: pd.DataFrame, n_bins: int = 10) -> float:
    """
    PSI trung bình trên tất cả feature.
    Bins được lấy theo quantile của test (ổn định khi test thay đổi scale).
    PSI ~ 0: rất giống; 0.1-0.25: shift vừa; >0.25: shift mạnh.
    """
    psis = []
    for col in X_test.columns:
        x_te = X_test[col].values
        x_tr = X_train[col].values
        x_te = x_te[~np.isnan(x_te)]
        x_tr = x_tr[~np.isnan(x_tr)]
        if len(x_te) == 0 or len(x_tr) == 0:
            continue

        # quantile breaks theo test
        qs = np.linspace(0, 1, n_bins+1)
        try:
            cuts = np.unique(np.quantile(x_te, qs))
            if len(cuts) < 3:
                continue
            # histogram theo breaks
            te_hist, _ = np.histogram(x_te, bins=cuts)
            tr_hist, _ = np.histogram(x_tr, bins=cuts)
            te_ratio = te_hist / max(1.0, te_hist.sum())
            tr_ratio = tr_hist / max(1.0, tr_hist.sum())
            # tránh log(0)
            te_ratio = np.clip(te_ratio, 1e-6, 1)
            tr_ratio = np.clip(tr_ratio, 1e-6, 1)
            psi = np.sum((tr_ratio - te_ratio) * np.log(tr_ratio / te_ratio))
            psis.append(psi)
        except Exception:
            continue

    return float(np.nanmean(psis)) if len(psis) else np.nan


In [14]:
# === Block ②: Time-slice weighting theo độ gần phân phối với Test ===
import numpy as np
from typing import Literal

def make_time_slices(X_train: pd.DataFrame, y_train: pd.Series, n_slices: int = 4):
    """Chia train theo thời gian thành n_slices đoạn bằng nhau."""
    n = len(X_train)
    edges = np.linspace(0, n, n_slices+1, dtype=int)
    slices = []
    for i in range(n_slices):
        a, b = edges[i], edges[i+1]
        Xi = X_train.iloc[a:b]
        yi = y_train.iloc[a:b]
        slices.append((a, b, Xi, yi))
    return slices

def slice_similarities(
    X_train: pd.DataFrame, 
    y_train: pd.Series, 
    X_test: pd.DataFrame,
    n_slices: int = 4,
    metric: Literal["ks","psi"] = "ks",
    psi_bins: int = 10
):
    """Tính độ gần (similarity) của từng slice với test (càng lớn càng "gần")."""
    sl = make_time_slices(X_train, y_train, n_slices)
    sims = []
    for (a,b,Xi,yi) in sl:
        if metric == "ks":
            d = ks_distance_matrix(Xi, X_test)         # nhỏ là gần
            sim = np.exp(-5.0 * d) if np.isfinite(d) else 0.0
        else:
            d = population_stability_index(Xi, X_test, n_bins=psi_bins)  # nhỏ là gần
            sim = np.exp(-2.0 * d) if np.isfinite(d) else 0.0
        sims.append(sim)
    sims = np.array(sims)
    if sims.sum() == 0:
        sims = np.ones_like(sims)
    sims = sims / sims.sum()  # chuẩn hoá để sum=1
    return sl, sims

def build_sample_weights_from_slices(
    X_train: pd.DataFrame, 
    y_train: pd.Series, 
    X_test: pd.DataFrame,
    n_slices: int = 4,
    metric: str = "ks",
    psi_bins: int = 10,
    add_time_decay: bool = True
):
    """
    Trọng số mẫu = (trọng số era theo similarity) * (class-balance) * (time-decay nhẹ)
    - similarity: exp(-λ * distance) với distance là KS trung bình hoặc PSI trung bình
    - class-balance: để cân bằng label trong train
    - time-decay: e^{α * t}, t∈[0,1]
    """
    sl, sims = slice_similarities(X_train, y_train, X_test, n_slices=n_slices, metric=metric, psi_bins=psi_bins)

    # map trọng số era -> từng sample
    w = np.zeros(len(X_train), dtype=float)
    pos = 0
    for k,(a,b,Xi,yi) in enumerate(sl):
        w[a:b] = sims[k]
    # time-decay (nhẹ)
    if add_time_decay:
        n = len(X_train)
        t = np.linspace(0, 1, n)
        td = np.exp(1.0 * t)      # α=1.0
        w = w * td

    # class-balance
    N = len(y_train); n0 = int((y_train==0).sum()); n1 = N - n0
    cw0 = N/(2.0*max(1,n0)); cw1 = N/(2.0*max(1,n1))
    class_w = np.where(y_train.values==1, cw1, cw0)

    w = w * class_w
    # chặn outlier & chuẩn hoá
    lo, hi = np.quantile(w, 0.02), np.quantile(w, 0.98)
    w = np.clip(w, lo, hi)
    w = w / np.median(w)
    return w


In [40]:
# === Block ③: Train & Evaluate với time-slice weighting ===
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report

# 1) Load & build base features
df_tr, df_te = load_raw("ACB")
data = build_base_features(df_tr, df_te, lag=30)
X_train, y_train = data["train"]
X_test,  y_test  = data["test"]

# 2) Tạo sample_weight theo era gần test (KS hoặc PSI)
#    - thử n_slices=4..6; metric="ks" (nhanh) hoặc "psi" (ổn định hơn)
w = build_sample_weights_from_slices(
    X_train, y_train, X_test,
    n_slices=5,
    metric="psi",     # đổi thành "psi" nếu bạn muốn dùng PSI
    psi_bins=10,
    add_time_decay=True
)

# 3) Train RF (class_weight đã nằm trong sample_weight → KHÔNG set class_weight thêm để tránh nhân đôi)
rf = RandomForestClassifier(
    n_estimators=10,
    max_depth=15,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features="sqrt",
    bootstrap=True,
    random_state=2025,
    n_jobs=-1
)
rf.fit(X_train, y_train, sample_weight=w)
y_pred = rf.predict(X_test)
bacc_rf = balanced_accuracy_score(y_test, y_pred)

print(f"[RF] Balanced accuracy (test): {bacc_rf:.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification report:\n", classification_report(y_test, y_pred, digits=4))


[Preprocess] Train: (1213, 124) | Test: (328, 124)
[Preprocess] Label dist train: Counter({0: 643, 1: 570}), test: Counter({0: 175, 1: 153})
[RF] Balanced accuracy (test): 0.5394
Confusion matrix:
 [[103  72]
 [ 78  75]]
Classification report:
               precision    recall  f1-score   support

           0     0.5691    0.5886    0.5787       175
           1     0.5102    0.4902    0.5000       153

    accuracy                         0.5427       328
   macro avg     0.5396    0.5394    0.5393       328
weighted avg     0.5416    0.5427    0.5420       328

